# Event Analysis Queries
* Ryan Kazmerik
* April 29, 2025

Proposed queries for basic event analysis, such as the number of events of a user, user event flow in a time window and additional queries that could be interesting to analyze the user behaviour.

In [57]:
import awswrangler as wr
import pandas as pd

In [58]:
DATABASE = "events_db"
S3_BUCKET = "s3://athena-query-results-806a5225/results/"

## Basic Queries
### Events by User
Let's see who the top 10 most active users are by querying to see the number of events by user

In [60]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        SELECT user_id, COUNT(*) AS total_events
        FROM events_db.events
        GROUP BY user_id
        ORDER BY total_events DESC;
    """
).head(10)

,user_id,total_events
0,601955,4
1,357199,4
2,761789,4
3,559832,4
4,407010,4
5,999632,4
6,149765,3
7,235870,3
8,978865,3
9,819066,3


### User Event Flow in a Time Window
Let's dig into the top user and see what activities they've been up to in the past 60 days

In [62]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH most_active_user AS (
            SELECT user_id
            FROM events_db.events
            GROUP BY user_id
            ORDER BY COUNT(*) DESC
            LIMIT 1
        )
        SELECT e.page, e.event_type, COUNT(*) AS event_count
        FROM events_db.events e
        JOIN most_active_user m ON e.user_id = m.user_id
        WHERE date_parse(e.event_date, '%Y-%m-%d') >= current_date - interval '60' day
        GROUP BY e.page, e.event_type
        ORDER BY event_count DESC;
    """
).head()

,page,event_type,event_count
0,products,view,1
1,pricing,click,1


## Additional Queries

### What Content is Driving Purchases?
* Objective: New Customer Acquisition
* Target Audience: Revenue Team 

One thing we're curious about is what's driving traffic to our purchase page? One way to investigate this may be to see what pages users were viewing before they decided to visit the purchase page to help understand what content is making users want to purchase.


In [63]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH user_journeys AS (
            SELECT user_id, event_type, event_timestamp, page,
            LAG(page) OVER (PARTITION BY user_id ORDER BY event_timestamp) AS previous_page
            FROM events_db.events
        ),
        product_visits AS (
            SELECT previous_page, COUNT(*) AS product_purchases
            FROM user_journeys
            WHERE page = 'products'
            AND previous_page != 'products'
            AND event_type = 'purchase'
            GROUP BY previous_page
        )
        SELECT previous_page, product_purchases
        FROM product_visits
        ORDER BY product_purchases DESC;
    """
).head(10)

,previous_page,product_purchases
0,login,2


### Which Users are Losing Interest?

* Objective : Customer Retention
* Target Audience : Customer Success Team

It may be helpful to identify users who used to use our app a lot, but their activity is dropping off. We could reach out to them to prevent them from churning from our service all together.

In [65]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH past_active_users AS (
            SELECT user_id, COUNT(*) as events_in_past_6m
            FROM events_db.events
            WHERE date_parse(event_date, '%Y-%m-%d') BETWEEN date_add('day', -180, current_date) AND date_add('day', -30, current_date)
            GROUP BY user_id
        ),
        recent_inactive_users AS (
            SELECT user_id, COUNT(*) as events_this_month
            FROM events_db.events
            WHERE date_parse(event_date, '%Y-%m-%d') >= date_add('day', -30, current_date)
            GROUP BY user_id
        )
        SELECT p.user_id, events_in_past_6m, events_this_month
        FROM past_active_users p
        LEFT JOIN recent_inactive_users r ON p.user_id = r.user_id
        WHERE r.user_id IS NULL
        ORDER BY events_in_past_6m DESC;
    """
).fillna(0).head(10)

,user_id,events_in_past_6m,events_this_month
0,814662,2,0
1,601643,2,0
2,999632,2,0
3,817215,2,0
4,846476,2,0
5,210129,2,0
6,357199,2,0
7,978865,2,0
8,544981,2,0
9,360862,2,0


### What Platform Drives the most Engagement?

* Objective : Channel Prioritization
* Target Audience : Product Team

We can look at which platform helps drive the most engagement with it's users. This might help inform the product team if one of our channels needs extra attention, or additional investment.

In [66]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH recent_events AS (
            SELECT device, date_trunc('month', event_timestamp) AS event_month
            FROM events_db.events
            WHERE date_parse(event_date, '%Y-%m-%d') >= current_date - interval '6' month
        ),
        monthly_event_counts AS (
            SELECT device, event_month, COUNT(*) AS monthly_events
            FROM recent_events
            GROUP BY device, event_month
        )
        SELECT device, ROUND(AVG(monthly_events), 2) AS avg_events_per_month
        FROM monthly_event_counts
        GROUP BY device
        ORDER BY avg_events_per_month DESC;
    """
).head(10)

,device,avg_events_per_month
0,Tablet,10.00
1,Desktop,8.50
2,Mobile,7.17


### Who's Having Trouble Logging In?

* Objective : Reduce Friction
* Target Audience : Support Team

Let's see if any users are consistently having trouble logging in in the past month, maybe we can proactively reach out to them and help them access our app.

In [70]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH failed_logins AS (
            SELECT user_id, device, browser, event_date
            FROM events_db.events
            WHERE page = 'login' AND event_type = 'error'
            AND date_parse(event_date, '%Y-%m-%d') >= current_date - interval '1' month
        )
        SELECT user_id, COUNT(*) AS failed_login_attempts
        FROM failed_logins
        GROUP BY user_id
        ORDER BY failed_login_attempts DESC;
    """
).head(10)

,user_id,failed_login_attempts


### These queries are simple but actionable and help us determine our next best action for improving our app.